In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/embeddings-lab/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# Exercises 05 · Evaluation & fusion
The three functions every retrieval system owner ends up writing.
Solutions: `solutions/ex05_solutions.ipynb`.

In [1]:
import numpy as np
from collections import Counter

## Task 1 — graded nDCG@k
`ndcg_at_k(gains, ideal_gains, k)` with `DCG = Σ gain_i / log2(i + 2)`.
Check: ranking with gains [3,2,0,1] vs ideal [3,2,1,0] → 0.98544.

In [2]:
def dcg(gains, k):
    return sum(g / np.log2(i + 2) for i, g in enumerate(gains[:k]))

def ndcg_at_k(gains, ideal_gains, k=10):
    ide = dcg(sorted(ideal_gains, reverse=True), k)
    return dcg(gains, k) / ide if ide > 0 else 0.0

assert np.isclose(ndcg_at_k([3, 2, 0, 1], [3, 2, 1, 0], k=10), 0.98544, atol=1e-4)
assert np.isclose(ndcg_at_k([3, 2, 1, 0], [3, 2, 1, 0], k=10), 1.0)
print("ndcg_at_k ✓")

ndcg_at_k ✓


## Task 2 — Reciprocal Rank Fusion
`rrf(rankings, k)`: score(d) = Σ over rankings 1/(k + rank(d) + 1), rank
0-based; return doc ids sorted by score. Check with k=1:
[a,b,c] + [c,a,b] → a: 1/2+1/3, c: 1/4+1/2, b: 1/3+1/4 → order a, c, b.

In [3]:
def rrf(rankings, k=60):
    sc = Counter()
    for ranked in rankings:
        for r, d in enumerate(ranked):
            sc[d] += 1.0 / (k + r + 1)
    return [d for d, _ in sc.most_common()]

assert rrf([["a", "b", "c"], ["c", "a", "b"]], k=1) == ["a", "c", "b"]
print("rrf ✓")

rrf ✓


## Task 3 — the two halves of a BM25 term score
`bm25_idf(N, df) = log(1 + (N − df + 0.5)/(df + 0.5))` and
`bm25_tf_norm(f, dl, avgdl, k1, b) = f·(k1+1) / (f + k1·(1 − b + b·dl/avgdl))`.
The second is the part worth internalizing: term-frequency **saturates** and
long documents are **penalized**.

In [4]:
def bm25_idf(N, df):
    return np.log(1 + (N - df + 0.5) / (df + 0.5))

def bm25_tf_norm(f, dl, avgdl, k1=1.5, b=0.75):
    return f * (k1 + 1) / (f + k1 * (1 - b + b * dl / avgdl))

assert np.isclose(bm25_idf(100, 10), np.log(1 + 90.5 / 10.5))
assert np.isclose(bm25_tf_norm(2, 100, 100), 10 / 7)
assert bm25_tf_norm(20, 100, 100) < 2.5 * bm25_tf_norm(1, 100, 100)   # saturation
assert bm25_tf_norm(2, 200, 100) < bm25_tf_norm(2, 50, 100)           # length penalty
print("bm25 components ✓")

bm25 components ✓


## Task 4 (open) — break the hybrid
In notebook 05, replace RRF with a weighted score sum
`α·z(bm25) + (1−α)·z(dense)` (z = standardize scores per query). Sweep α.
Why does RRF usually win without tuning? (Hint: score scales vs rank scales.)